# Vorbereitungen für alle Kurs-Repositories

Dieses Notebook richtet eine Python-Umgebung für alle drei Teile des Seminars ein:

| Repository | Inhalt |
|---|---|
| `intro-python-basics-de` | Python-Grundlagen |
| `intro-data-mining-de` | Pattern Mining, Klassifikation, Clusteranalyse |
| `intro-text-mining-de` | Text Mining (Skripte 01-09) |

Es fasst die beiden Notebooks `00-preparations.ipynb` aus `intro-data-mining-de` und `intro-text-mining-de` zusammen und ergänzt, was die Skripte beim ersten Lauf sonst selbst herunterladen (NLTK-Stoppwörter, Sentence-Transformer-Modell, Kaggle-Datensätze). Es liegt im Repository `intro-python-basics-de` und erwartet, dass die beiden anderen Repositories im selben übergeordneten Ordner liegen:

```
projects/
├── intro-python-basics-de/   ← dieses Notebook
├── intro-data-mining-de/
└── intro-text-mining-de/
```

Es muss nur einmal pro Python-Umgebung ausgeführt werden.

| Schritt | Was | Größe |
|---|---|---|
| 1 | Python-Version prüfen | - |
| 2 | Pakete installieren | ca. 1-2 GB (vor allem `torch`) |
| 3 | spaCy-Modelle `de_core_news_sm/md/lg` | ca. 700 MB |
| 4 | NLTK-Stoppwörter | < 1 MB |
| 5 | Sentence-Transformer-Modell | ca. 0,5 GB |
| 6 | Kaggle-Datensätze (Data Mining 01 und 03) | wenige MB |
| 7 | Text-Mining-Korpus | ca. 1 GB Download, ca. 100 MB gespeichert |
| 8 | API-Schlüssel (nur Text Mining 07 und 09) | - |
| 9 | Alles prüfen | - |

## 1. Python-Version und Ordner prüfen

Die Skripte brauchen Python 3.10 oder neuer. Die Zelle zeigt auch, welches Python dieser Kernel verwendet - genau dort werden die Pakete installiert - und ob die drei Repositories im übergeordneten Ordner liegen.

In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
assert sys.version_info >= (3, 10), "Die Skripte brauchen Python 3.10 oder neuer."

# Ordner, der die drei Repositories enthält (normalerweise der übergeordnete Ordner)
ROOT = Path.cwd()
if not (ROOT / "intro-text-mining-de").is_dir():
    ROOT = ROOT.parent
print("Ordner:", ROOT)
REPOS = ["intro-python-basics-de", "intro-data-mining-de", "intro-text-mining-de"]
for repo in REPOS:
    print(f"{repo:25} {'gefunden' if (ROOT / repo).is_dir() else 'FEHLT'}")

## 2. Pakete installieren

`%pip install` installiert in genau die Umgebung dieses Kernels. Die Liste ist die Vereinigung der Pakete aus den drei READMEs:

| Paket | Verwendet in |
|---|---|
| `pandas`, `numpy`, `matplotlib` | allen Teilen |
| `openpyxl` | Python-Grundlagen (Excel-Datei `pwt100.xlsx`) |
| `kagglehub`, `mlxtend`, `scipy`, `seaborn`, `geopandas`, `pyproj`, `networkx` | Data Mining |
| `scikit-learn`, `torch` | Data Mining und Text Mining |
| `pyarrow`, `nltk`, `spacy`, `gensim`, `sentence-transformers`, `umap-learn` | Text Mining |
| `anthropic`, `mistralai` | Text Mining 07 und 09 (LLM-APIs) |
| `jupyterlab` | Notebooks ausführen |
| `ipywidgets` | Fortschrittsbalken in Jupyter (z. B. beim Laden von Modellen) |

In [ ]:
%pip install anthropic geopandas gensim ipywidgets jupyterlab kagglehub matplotlib mistralai mlxtend networkx nltk numpy openpyxl pandas pyarrow pyproj scikit-learn scipy seaborn sentence-transformers spacy torch umap-learn

Die Installation dauert beim ersten Mal einige Minuten, vor allem wegen `torch`. **Danach den Kernel neu starten** (*Kernel → Restart Kernel*) und mit Schritt 3 weitermachen - Schritt 1 einmal erneut ausführen, damit `ROOT` wieder gesetzt ist.

Unter Windows und macOS installiert pip `torch` für die CPU. Wer unter Windows eine NVIDIA-Grafikkarte nutzen möchte, folgt https://pytorch.org/get-started/locally/. Nötig ist das für die Kurs-Daten nicht.

## 3. Sprachmodelle von spaCy

Text Mining verwendet drei deutsche Modelle: `sm` (Skript 02), `md` (die meisten Skripte) und `lg` (Wortvektoren in Skript 06). Die Zelle lädt nur fehlende Modelle und verwendet dafür das Python des Kernels (`sys.executable`).

In [ ]:
import importlib.util
import subprocess
import sys

for model in ["de_core_news_sm", "de_core_news_md", "de_core_news_lg"]:
    if importlib.util.find_spec(model) is None:
        print(f"lade {model} ...")
        subprocess.run([sys.executable, "-m", "spacy", "download", model], check=True)
    else:
        print(f"{model} ist schon installiert")

## 4. NLTK-Stoppwörter

Text Mining 01 verwendet die deutsche Stoppwortliste von NLTK. Das Skript lädt sie auch selbst; hier liegt sie danach schon bereit.

In [ ]:
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
print(len(stopwords.words("german")), "deutsche Stoppwörter")

## 5. Sentence-Transformer-Modell

Text Mining 06, 07 und 09 verwenden `paraphrase-multilingual-MiniLM-L12-v2` vom Hugging Face Hub. Die Zelle lädt es einmal in den lokalen Cache (`~/.cache/huggingface`) und testet es an zwei Sätzen.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
emb = encoder.encode(["Der Nationalrat beschließt das Budget.", "Parliament passes the budget."])
print("Embedding-Dimension:", emb.shape[1])
print("Ähnlichkeit:", round(float(encoder.similarity(emb[:1], emb[1:])[0, 0]), 3))
del encoder

## 6. Kaggle-Datensätze (Data Mining)

Data Mining 01 und 03 laden je einen öffentlichen Datensatz mit `kagglehub`. Der Download braucht kein Kaggle-Konto; `kagglehub` speichert die Dateien in `~/.cache/kagglehub`, und die Skripte finden sie dort.

In [ ]:
import kagglehub

for dataset in ["lissetteg/ecommerce-dataset/versions/1",   # Skript 01, Pattern Mining
                "koki25ando/scotch-whisky-dataset"]:        # Skript 03, Clusteranalyse
    path = kagglehub.dataset_download(dataset)
    print(f"{dataset:42} -> {path}")

## 7. Text-Mining-Korpus

`textmining.py` lädt das One Million Posts Corpus (Der Standard) und den deutschen Teil von MLSUM (Süddeutsche Zeitung) herunter, rund 1 GB, und speichert die benötigten Teile als drei Parquet-Dateien (knapp 100 MB) in `intro-text-mining-de/data/`. Das dauert einige Minuten; sind die Dateien schon da, passiert nichts. Soll der Ordner woanders liegen, vorher die Umgebungsvariable `TEXTMINING_DATA` setzen. Lizenzen und Quellen stehen in der README von `intro-text-mining-de`.

In [ ]:
TM_DIR = ROOT / "intro-text-mining-de"
if str(TM_DIR) not in sys.path:
    sys.path.insert(0, str(TM_DIR))

import textmining as tm

tm.prepare_data()
corpus, posts = tm.load_corpus(), tm.load_posts()
print(f"{len(corpus):,} Schlagzeilen, {len(posts):,} annotierte Leserkommentare")
print(corpus["outlet"].value_counts().to_string())
print("Daten in:", tm.DATA)

## 8. API-Schlüssel (nur Text Mining 07 und 09)

Skript 07 nutzt Claude (Anthropic), Skript 09 Claude und Mistral. Die Schlüssel gehören in `intro-text-mining-de/api-keys.env`; die Datei ist in `.gitignore` eingetragen. Fehlt sie, legt die Zelle sie als Kopie der Vorlage an - danach die Schlüssel dort eintragen und die Zelle erneut ausführen. Ohne Schlüssel laufen beide Skripte trotzdem durch (mit `RUN_LLM = False`). Die Zelle zeigt nur, welche Schlüssel gefunden wurden, nicht die Schlüssel selbst.

In [ ]:
import shutil

keys_file = TM_DIR / "api-keys.env"
if not keys_file.exists():
    shutil.copy(TM_DIR / "api-keys.example.env", keys_file)
    print(f"{keys_file} angelegt - bitte Schlüssel eintragen")

tm.load_api_keys()

## 9. Alles prüfen

Lädt alle Pakete und Modelle und gibt ihre Versionen aus. Ein `ModuleNotFoundError` oder `OSError` heißt: Paket oder Modell fehlt - meist hilft ein Neustart des Kernels nach Schritt 2 oder 3.

In [ ]:
from importlib.metadata import version

import anthropic, geopandas, gensim, ipywidgets, kagglehub, matplotlib, mistralai, mlxtend, networkx, nltk, numpy
import openpyxl, pandas, pyarrow, pyproj, scipy, seaborn, sentence_transformers, sklearn, spacy, torch, umap

for paket in ["anthropic", "geopandas", "gensim", "ipywidgets", "jupyterlab", "kagglehub", "matplotlib", "mistralai",
              "mlxtend", "networkx", "nltk", "numpy", "openpyxl", "pandas", "pyarrow", "pyproj",
              "scikit-learn", "scipy", "seaborn", "sentence-transformers", "spacy", "torch", "umap-learn"]:
    print(f"{paket:22} {version(paket)}")

for modell in ["de_core_news_sm", "de_core_news_md", "de_core_news_lg"]:
    print(f"{modell:22} {spacy.load(modell).meta['version']}")

print("Grafikkarte (CUDA):    ", "ja" if torch.cuda.is_available() else "nein, die Skripte laufen auf der CPU")